# H2 cross-ablation: do the P and T circuits cause each other's override? (Qwen2.5-VL-3B)

Owner: **@SohamRShinde**. Design and reading rule pre-registered in
`docs/DECISIONS.md` **D73** before this notebook ran.

The overlap result (J_PT = 0.312, 5 of 16 heads shared) says the two circuits
are *selected* partly alike. This asks whether they are *causally* shared:

| test | ablate | on | question |
|---|---|---|---|
| **P->T** (primary) | H_P, the 9 P heads | T objects | does knocking out the prior-override circuit defeat the note attack? |
| **T->P** (primary) | H_T, the 12 T heads | P objects | does knocking out the note circuit defeat the prompted prior? |
| P->P, T->T | own circuit | own objects | reference (in-sample) |
| shared, P-only, T-only | the decomposition | both | is the transfer carried only by the 5 shared heads? |

Each effect is compared with **100 random head sets with the same number of
heads in each layer**, drawn from heads neither circuit uses (D73 (5)).
Flips are Lietzow's pairwise definition (D67). Zero ablation of the `o_proj`
input slice at the last token, as everywhere in this repo.

**Phases**, each checkpointed one JSON line per object (D44):

1. **Zero ablation** (~5 h): P's 73 objects, then T's 244, ~400 configurations each.
2. **Mean ablation** (~1 h): the two primary tests again, heads set to their mean activation.
3. **Clean image** (~15 min): does ablation break plain colour naming?
4. **Save.**

**Runtime ~6.5 h. Run it as Save Version -> Save & Run All (Commit).**
Section 2 times the real steps and prints the estimate.

**To resume after a cut-off:** attach the previous version's output as an
input dataset and re-run everything; section 2 copies its `cross_ablation/ckpt/`
folder back and the loops skip finished objects.

**Output:** `/kaggle/working/cross_ablation/`. Download only the **two** files
the last cell names. Everything under `ckpt/` is checkpoints and cached
activations (~100 MB), not results.

Settings: GPU T4, Internet ON, Kaggle secret `HF_TOKEN` ticked under Add-ons -> Secrets.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
import importlib.util as _ilu
_need = {}
for _mod, _names in {
    "circuit_overlap.cross_ablation": ["build_plan", "plan_cost", "ablate_heads_last_token",
                                       "analyse_cross_ablation", "source_flips"],
    "circuit_overlap.t_patching": ["render_t_pair", "LAYOUT", "load_patching_records"],
    "circuit_overlap.patching": ["LastTokenRunner", "capture_last_token_o_proj_inputs",
                                 "patch_head_last_token"],
    "circuit_overlap.replication": ["lietzow_rows", "color_token_sets", "run_cached"],
    "circuit_overlap.t_screen": ["PRIOR_PROMPT", "VISUAL_PROMPT", "load_notes_config",
                                 "color_first_token_ids"],
}.items():
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = ["module"]
        continue
    _m = importlib.import_module(_mod)
    _gone = [n for n in _names if not hasattr(_m, n)]
    if _gone:
        _need[_mod] = _gone
for _f in ("results/replication_patching_qwen25vl3b.json", "results/t_patching_qwen25vl3b.json",
           "results/t_patching_records_qwen25vl3b.jsonl.gz",
           "results/replication_knockout_qwen25vl3b.json"):
    if not os.path.exists(f"{REPO_DIR}/{_f}"):
        _need[_f] = ["file"]
if _need:
    raise RuntimeError(f"checkout {COMMIT} is missing {_need}. "
                       "Push the local commits to origin/main, then re-run this cell.")
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). Installing it into a
# kernel that already imported transformers does nothing: restart and re-run.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
from circuit_overlap.t_screen import native_bf16
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
# Not torch.cuda.is_bf16_supported(): it says True on a T4 (emulation).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    # Secrets are attached PER NOTEBOOK: one on your account is not enough.
    raise RuntimeError(
        "HF_TOKEN is not attached to this notebook. In the editor: Add-ons -> "
        "Secrets -> tick the checkbox next to HF_TOKEN (exact label), then "
        "restart the session and run from cell 1. It must be ticked before "
        "Save & Run All, or the background run fails here."
    ) from exc

In [ ]:
from circuit_overlap.t_screen import ScreenConfig, VISUAL_PROMPT, load_model
config = ScreenConfig()
model, processor = load_model(config)        # 4-bit nf4, double quant, fp16 compute on a T4 (D10)
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

text_config = model.model.language_model.config
n_layers = text_config.num_hidden_layers
n_heads = text_config.num_attention_heads
head_dim = getattr(text_config, "head_dim", text_config.hidden_size // n_heads)
print(f"{n_layers} layers x {n_heads} heads, head_dim={head_dim}")
assert (n_layers, n_heads) == (36, 16), "not the geometry P was discovered on"

## 1b. [VERIFY] Image size and speed

Same check as `replication_patching.ipynb`: the image token budget must be
~1,280 (pinned area, as Lietzow), and a forward pass must take seconds. The
input builder is identical to P's, so the two conditions are tokenised alike.

In [ ]:
import time, torch
from datasets import load_dataset
from transformers import AutoProcessor

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_probe = ds[0]["original_image"].convert("RGB")

def build_inputs(image, prompt):
    messages = [{"role": "user",
                 "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[image], return_tensors="pt").to(model.device)

def n_image_tokens(image):
    t, h, w = build_inputs(image, "x")["image_grid_thw"][0].tolist()
    return t * h * w // 4                    # 2x2 patch merge -> LLM tokens

EXPECTED = config.pixels // (28 * 28)        # 1280
n_img = n_image_tokens(_probe)
print(f"image {_probe.size} -> {n_img} image tokens (expected ~{EXPECTED})")
if n_img > 1.2 * EXPECTED:
    processor = AutoProcessor.from_pretrained(
        config.checkpoint, use_fast=True,
        size={"shortest_edge": config.pixels, "longest_edge": config.pixels},
    )
    n_img = n_image_tokens(_probe)
    print(f"rebuilt the processor with size=... -> {n_img} image tokens")
assert n_img <= 1.2 * EXPECTED, "image token budget is still wrong -- do not run the loops"

assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 and restart"
_inp = build_inputs(_probe, VISUAL_PROMPT.format(object="banana"))
for attempt in ("warm-up", "timed"):
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        model(**_inp, use_cache=False)
    torch.cuda.synchronize()
    print(f"{attempt} forward pass: {time.time() - t0:.1f}s")

## 2. The plan, the objects, and checks of the ablation

The head sets and the random draws come from `cross_ablation.build_plan`,
which reads the P and T discovery results committed in `results/` (D73 (1),
(5)); nothing here is typed by hand. The objects are exactly those patched in
discovery, with the same inputs: P on the counterfactual image with the
Prior (source) and Visual (target) prompts; T on the original image with the
incongruent (source) and neutral (target) `on_object_small` notes.

The checks stop the notebook if the ablation is wrong: the cached path must
equal a full forward pass under ablation; stacked per-head hooks must equal
Kush's one-hook-per-layer knockout; the result must match Kush's recorded
knockout of the same 9 heads on the same object.

In [ ]:
import glob, gzip, inflect, json, math, pathlib, shutil
from collections import Counter
from circuit_overlap.replication import lietzow_rows, color_token_sets, run_cached
from circuit_overlap.t_screen import PRIOR_PROMPT, color_first_token_ids, load_notes_config
from circuit_overlap.patching import LastTokenRunner, capture_last_token_o_proj_inputs
from circuit_overlap.t_patching import LAYOUT, render_t_pair, load_patching_records
from circuit_overlap.cross_ablation import build_plan, plan_cost, ablate_heads_last_token

WORK = pathlib.Path("/kaggle/working/cross_ablation")
CKPT = WORK / "ckpt"            # checkpoints and cached activations: NOT results
(CKPT / "acts").mkdir(parents=True, exist_ok=True)

# Resume: bring back checkpoints from an attached previous run's output.
found = sorted(glob.glob("/kaggle/input/**/cross_ablation/ckpt", recursive=True))
if found:
    shutil.copytree(found[0], CKPT, dirs_exist_ok=True)
    print(f"resuming from {found[0]}: {sorted(p.name for p in CKPT.glob('*.jsonl'))}")

_eng = inflect.engine()
rows, dropped = lietzow_rows(ds, _eng)
row_by_i = {r["i"]: r for r in rows}
vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
ambiguous = set(collisions)
print(len(rows), "objects |", len(vocab), "colours | sharing a first token (D26):", collisions)

notes_cfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")
spec = next(s for s in notes_cfg["specs"] if s.name == LAYOUT)
neutral_words = notes_cfg["neutral_words"]

p_res = json.load(open(f"{REPO_DIR}/results/replication_patching_qwen25vl3b.json"))["result"]
t_res = json.load(open(f"{REPO_DIR}/results/t_patching_qwen25vl3b.json"))["result"]
plan = build_plan(p_res, t_res, n_heads_per_layer=n_heads)
configs = {k: [tuple(h) for h in v] for k, v in plan["configs"].items()}
for k, v in plan["sets"].items():
    print(f"  {k:7s} ({len(v):2d}): {', '.join(v)}")
print(len(plan["excluded_from_draws"]), "classified heads excluded from the random draws")

# Discovery argmaxes: an object must give the same unablated answers here (D73 (2)).
p_disc = {r["i"]: (r["next_token_idx_prior"], r["next_token_idx_visual"])
          for r in p_res["records"] if r["status"] == "patched"}
t_disc = {r["i"]: (r["next_token_idx_prior"], r["next_token_idx_visual"])
          for r in load_patching_records(f"{REPO_DIR}/results/t_patching_records_qwen25vl3b.jsonl.gz")}
todo = {"P": [row_by_i[i] for i in p_disc], "T": [row_by_i[i] for i in t_disc]}
disc = {"P": p_disc, "T": t_disc}
clean_ids = sorted(set(p_disc) | set(t_disc))
print(f"P {len(todo['P'])} objects, T {len(todo['T'])}, clean {len(clean_ids)} (expect 73, 244, 277)")

def inputs(cond, r):
    """(source inputs, target inputs) for one object, as in its discovery run."""
    row = ds[r["i"]]
    if cond == "P":
        image = row["counterfact_image"].convert("RGB")
        return (build_inputs(image, PRIOR_PROMPT.format(a_object=_eng.a(r["name"]))),
                build_inputs(image, VISUAL_PROMPT.format(object=r["name"])))
    pr = render_t_pair(row["original_image"], row["counterfact_image"],
                       r["attack"], neutral_words, spec)
    prompt = VISUAL_PROMPT.format(object=r["name"])
    return build_inputs(pr["incongruent"], prompt), build_inputs(pr["neutral"], prompt)

def pair(logits, a):
    """[logit of the source run's answer, logit of the target run's answer] (D67)."""
    return [round(float(logits[a[0]]), 4), round(float(logits[a[1]]), 4)]

def run_configs(source, target, entries, a, key_prefix="", values=None):
    """Each configuration's ablated logit pair, on the source run and, if asked, the target."""
    out = {}
    for name, runs in entries:
        for kind, run in [("prior", source)] + ([("visual", target)] if runs == "both" else []):
            with ablate_heads_last_token(model, configs[name], head_dim, values):
                out[f"{key_prefix}{name}/{kind}"] = pair(run.logits(), a)
    return out

def finite(d):
    return all(math.isfinite(x) for v in d.values() if isinstance(v, list) for x in v)

In [ ]:
from contextlib import contextmanager

@contextmanager
def kush_knockout(heads):
    """replication_flip_rate.ipynb's head_knockout: one hook per layer."""
    by_layer = {}
    for layer, head in heads:
        by_layer.setdefault(layer, []).append(head)
    layers = model.model.language_model.layers
    handles = []
    for layer, layer_heads in by_layer.items():
        def pre_hook(module, args, layer_heads=tuple(layer_heads)):
            x = args[0].clone()
            for h in layer_heads:
                x[:, -1, h * head_dim:(h + 1) * head_dim] = 0
            return (x, *args[1:])
        handles.append(layers[layer].self_attn.o_proj.register_forward_pre_hook(pre_hook))
    try:
        yield
    finally:
        for handle in handles:
            handle.remove()

VERIFY = {}
# Runners are built OUTSIDE any hook: an ablated prefix run would corrupt the cache.
r0 = todo["P"][0]
t0_ = time.time()
source0, target0 = (LastTokenRunner(model, x) for x in inputs("P", r0))
prefill = (time.time() - t0_) / 2
plain = source0.logits()
a0 = (int(plain.argmax()), int(target0.logits().argmax()))
assert a0 == tuple(p_disc[r0["i"]]), f"{r0['name']}: answers differ from discovery {a0} vs {p_disc[r0['i']]}"

# [VERIFY] 1. the cached path equals a full forward pass under ablation of H_P.
with ablate_heads_last_token(model, configs["P"], head_dim):
    VERIFY["cached_vs_full_ablated"] = source0.check_against_full_forward(atol=1.0)
# [VERIFY] 2. stacked per-head hooks == Kush's one-hook-per-layer knockout.
with kush_knockout(configs["P"]):
    ref = source0.logits()
with ablate_heads_last_token(model, configs["P"], head_dim):
    ours = source0.logits()
VERIFY["stacked_vs_kush_hook"] = float((ref - ours).abs().max())
assert VERIFY["stacked_vs_kush_hook"] < 1e-3, VERIFY
# [VERIFY] 3. an empty ablation changes nothing; a zero "mean" equals zero ablation.
with ablate_heads_last_token(model, [], head_dim):
    assert torch.equal(source0.logits(), plain)
zeros = {h: torch.zeros(head_dim) for h in configs["P"]}
with ablate_heads_last_token(model, configs["P"], head_dim, values=zeros):
    VERIFY["values_path_vs_zero"] = float((source0.logits() - ours).abs().max())
assert VERIFY["values_path_vs_zero"] < 1e-3, VERIFY
# [VERIFY] 4. matches Kush's recorded knockout of the same 9 heads on this object.
knock = {r["i"]: r for r in json.load(open(f"{REPO_DIR}/results/replication_knockout_qwen25vl3b.json"))["result"]["records"]}
k0 = knock.get(r0["i"])
if k0 is not None:
    a_k = (k0["a_prior"], k0["a_visual"])
    with ablate_heads_last_token(model, configs["P"], head_dim):
        mine = {"prior": pair(source0.logits(), a_k), "visual": pair(target0.logits(), a_k)}
    VERIFY["vs_kush_record"] = {"ours": mine, "kush": {"prior": k0["grp_promoting/prior"],
                                                       "visual": k0["grp_promoting/visual"]}}
    print("vs Kush's record:", VERIFY["vs_kush_record"], "(expect agreement to ~0.1, fp16)")
# [VERIFY] 5. a T object: same answers as discovery, cached path fine under H_T.
r1 = todo["T"][0]
source1, target1 = (LastTokenRunner(model, x) for x in inputs("T", r1))
a1 = (int(source1.logits().argmax()), int(target1.logits().argmax()))
assert a1 == tuple(t_disc[r1["i"]]), f"{r1['name']}: answers differ from discovery"
with ablate_heads_last_token(model, configs["T"], head_dim):
    VERIFY["cached_vs_full_ablated_T"] = target1.check_against_full_forward(atol=1.0)
print("checks passed:", {k: v for k, v in VERIFY.items() if k != "vs_kush_record"})

# Time an ablated step and estimate the whole run.
torch.cuda.synchronize(); t0_ = time.time()
for _ in range(10):
    with ablate_heads_last_token(model, configs["T"], head_dim):
        source1.logits()
torch.cuda.synchronize()
step = (time.time() - t0_) / 10
cost = plan_cost(plan, {c: len(v) for c, v in todo.items()}, len(clean_ids))
hours = {k: cost[k] * step / 3600 for k in ("zero", "mean", "clean")}
hours["prefills"] = cost["prefills"] * prefill / 3600
VERIFY.update(step_ms=step * 1000, prefill_s=prefill)
print(f"ablated step {step * 1000:.0f} ms, prefill {prefill:.1f} s ->",
      {k: f"{v:.1f} h" for k, v in hours.items()}, f"total ~{sum(hours.values()):.1f} h")
del source0, target0, source1, target1

## 3. Phase 1: zero ablation (about 5 h)

For each object: the two unablated runs (their answers must equal
discovery's, or the object is dropped as `drift`), their last-token
activations saved for phase 2's means, then every zero-ablation
configuration of the plan. P's 73 objects first, then T's 244.

In [ ]:
def zero_object(cond):
    def fn(r):
        out = {"condition": cond, "name": r["name"]}
        src_in, tgt_in = inputs(cond, r)
        if src_in["input_ids"].shape != tgt_in["input_ids"].shape:
            return {"status": "shape_mismatch", **out}
        source, target = LastTokenRunner(model, src_in), LastTokenRunner(model, tgt_in)
        with capture_last_token_o_proj_inputs(model) as acts_s:
            logits_s = source.logits()
        with capture_last_token_o_proj_inputs(model) as acts_t:
            logits_t = target.logits()
        a = (int(logits_s.argmax()), int(logits_t.argmax()))
        out.update(a_prior=a[0], a_visual=a[1])
        if a != tuple(disc[cond][r["i"]]):
            return {"status": "drift", **out, "discovery": list(disc[cond][r["i"]])}
        torch.save({"source": torch.stack([acts_s[l] for l in range(n_layers)]).half().cpu(),
                    "target": torch.stack([acts_t[l] for l in range(n_layers)]).half().cpu()},
                   CKPT / "acts" / f"{cond}_{r['i']}.pt")
        out.update(run_configs(source, target, plan["zero"][cond], a))
        out["status"] = "ok" if finite(out) else "numerics_failed"      # plan risk 7
        return out
    return fn

from circuit_overlap.cross_ablation import source_flips
zero = {}
for cond in ("P", "T"):
    zero[cond] = run_cached(CKPT / f"zero_{cond}.jsonl", todo[cond], zero_object(cond), progress_every=5)
    ok = [r for r in zero[cond] if r["status"] == "ok"]
    print(cond, dict(Counter(r["status"] for r in zero[cond])))
    print(f"  own circuit, preliminary: {source_flips(ok, cond).mean():.1%} source flips"
          + ("   (P->P should be near Kush's 89.0%)" if cond == "P" else ""))

## 4. Phase 2: mean ablation (about 1 h)

D73 (10): each ablated head's slice is set to its **mean** last-token
activation over both runs of all of that condition's objects (from phase 1),
instead of zero. Only the two primary tests are repeated: H_P and the
H_P-matched draws on T, H_T and the H_T-matched draws on P.

In [ ]:
def condition_means(cond):
    total, n = None, 0
    for r in zero[cond]:
        if r["status"] != "ok":
            continue
        path = CKPT / "acts" / f"{cond}_{r['i']}.pt"
        if not path.exists():
            raise FileNotFoundError(
                f"{path} is missing: attach the previous run's output including ckpt/acts/, "
                f"or delete {CKPT}/zero_{cond}.jsonl and re-run phase 1")
        d = torch.load(path)
        for run in ("source", "target"):
            x = d[run].float()
            total = x if total is None else total + x
            n += 1
    return total / n                     # (n_layers, n_heads * head_dim)

def mean_object(cond, values):
    a_of = {r["i"]: (r["a_prior"], r["a_visual"]) for r in zero[cond] if r["status"] == "ok"}
    def fn(r):
        if r["i"] not in a_of:
            return {"status": "not_ok_in_phase_1", "condition": cond}
        source, target = (LastTokenRunner(model, x) for x in inputs(cond, r))
        out = {"condition": cond,
               **run_configs(source, target, plan["mean"][cond], a_of[r["i"]],
                             key_prefix="mean:", values=values)}
        out["status"] = "ok" if finite(out) else "numerics_failed"
        return out
    return fn

mean = {}
for cond in ("P", "T"):
    m = condition_means(cond).to(model.device)
    needed = {h for name, _ in plan["mean"][cond] for h in configs[name]}
    values = {(l, h): m[l, h * head_dim:(h + 1) * head_dim] for l, h in needed}
    mean[cond] = run_cached(CKPT / f"mean_{cond}.jsonl", todo[cond], mean_object(cond, values),
                            progress_every=10)
    print(cond, dict(Counter(r["status"] for r in mean[cond])))

## 5. Phase 3: clean image (about 15 min)

D73 (9), collateral: the original image, the Visual prompt, no note. Is the
top-1 still one of the object's real colours with each set ablated? On every
object in P or T; objects whose colours share a first token (D26) are skipped.

In [ ]:
def clean_object(r):
    out = {"condition": "clean", "name": r["name"]}
    if ambiguous & ({r["attack"]} | set(r["correct"])):
        return {"status": "ambiguous", **out}
    image = ds[r["i"]]["original_image"].convert("RGB")
    run = LastTokenRunner(model, build_inputs(image, VISUAL_PROMPT.format(object=r["name"])))
    _, real = color_token_sets(r, color_ids)
    for name in plan["clean"]:
        with ablate_heads_last_token(model, configs[name], head_dim):
            logits = run.logits()
        if not torch.isfinite(logits).all():
            return {"status": "numerics_failed", **out}
        out[f"{name}/correct"] = int(logits.argmax()) in real
    out["status"] = "ok"
    return out

clean = run_cached(CKPT / "clean.jsonl", [row_by_i[i] for i in clean_ids], clean_object,
                   progress_every=25)
print(dict(Counter(r["status"] for r in clean)),
      f"| correct unablated: {sum(r.get('none/correct', False) for r in clean)}")

## 6. Save

One record per object and condition, phases merged; an object counts only if
every phase is `ok` for it. The numbers printed here are **preliminary**: the
reported ones, with intervals and D73's wording, come from
`scripts/analyze_cross_ablation.py` run on the two files saved below.

In [ ]:
from circuit_overlap.cross_ablation import analyse_cross_ablation
from circuit_overlap.provenance import result_envelope, write_result

merged = {}
for cond in ("P", "T"):
    by_i = {r["i"]: r for r in mean[cond]}
    merged[cond] = []
    for r in zero[cond]:
        rec = {**r, "condition": cond}
        m = by_i.get(r["i"], {})
        rec.update({k: v for k, v in m.items() if k.startswith("mean:")})
        if rec["status"] == "ok" and m.get("status") != "ok":
            rec["status"] = f"phase2_{m.get('status', 'missing')}"
        merged[cond].append(rec)

prelim = analyse_cross_ablation(plan, merged, clean, n_boot=200)
for name, t in prelim["primary"].items():
    print(f"PRELIMINARY {name}: {t['count']}/{t['n']} = {t['observed']:.1%} vs random "
          f"{t['null_mean']:.1%} (max {t['null_max']:.1%}), p = {t['p']:.4f} -> {t['wording']}")

summary = {
    "plan": plan,
    "status": {c: dict(Counter(r["status"] for r in merged[c])) for c in merged},
    "clean_status": dict(Counter(r["status"] for r in clean)),
    "verify": VERIFY,
    "records_file": "cross_ablation_records_qwen25vl3b.jsonl.gz",
}
envelope = result_envelope(
    "cross_ablation", summary,
    model="Qwen2.5-VL-3B-Instruct", checkpoint=config.checkpoint, seed=config.seed,
    prompt=f"P: prior={PRIOR_PROMPT!r} vs visual={VISUAL_PROMPT!r}; T: visual, incongruent vs neutral note",
    n=sum(len(v) for v in merged.values()), repo_dir=REPO_DIR, decision="D73",
    quantisation="4-bit nf4, double quant",
    compute_dtype="bfloat16" if native_bf16(torch.cuda.get_device_capability(0)) else "float16",
    gpu=torch.cuda.get_device_name(0), image_tokens=n_img, layout=LAYOUT,
    ablation="o_proj input slice at last token, zero (phase 1, 3) or condition mean (phase 2); "
             "cached last-token path",
)
print(write_result(WORK / "cross_ablation_qwen25vl3b.json", envelope))
with gzip.open(WORK / "cross_ablation_records_qwen25vl3b.jsonl.gz", "wt") as f:
    for cond in ("P", "T"):
        for r in merged[cond]:
            f.write(json.dumps(r) + "\n")
    for r in clean:
        f.write(json.dumps(r) + "\n")
print("\nDownload ONLY these two into results/ (ckpt/ is checkpoints, not results):")
for name in ("cross_ablation_qwen25vl3b.json", "cross_ablation_records_qwen25vl3b.jsonl.gz"):
    print(f"  {name:44s} {(WORK / name).stat().st_size / 1e6:6.2f} MB")